# DemoClinic：Web Demo 体验与修复工作流

上传网站源码 ZIP，填写项目要求，由 Agent 操作浏览器检查功能，再按你的选择修复问题。

**流程：上传与启动 → 分析任务 → 浏览器体验 → 复现与报告 → 选择修复 → 复测 → 导出。**

只需要体验报告时，第10步结束即可。需要修复时继续第11–13步，最后运行第14步清除密钥。

分析、体验、修复分别配置模型和 API Key。报告由 Python 根据记录生成。项目和运行结果保存在本机，模型调用使用你配置的 API。


## 1 · 解包程序，建立运行目录

本单元格将 Notebook 内打包的通用程序解压到：

`DemoClinic运行目录/implementation`

其中包括：

- `workflow_engine.py`：安排测试、修复、复测和报告生成。
- `task_queue.py`：生成分层任务卡，按依赖分发体验任务。
- `project_submission.py`：显示上传表单，解压与导出项目。
- `experience_skill.py`：按需读取检查规范。
- `browser_worker.py`：操作浏览器、截图和执行检查。
- `browser_capabilities.py`：扩展浏览器工具、会话、环境模拟和覆盖记录。
- `体验Agent检查维度.md`：检查标准和执行规范；同文件夹已存在的版本不会被覆盖。
- `requirements.txt`：记录需要安装的 Python 工具。

本步骤不包含待测项目。目标项目将在后续配置中指定。

运行本单元格只会解包和加载程序，不会调用模型或开始测试。重复运行会覆盖解包目录中的同名程序文件。

In [ ]:
from pathlib import Path
import base64, io, zipfile, sys, os, importlib, json, datetime, shutil
from IPython.display import display, HTML, Markdown

WORK_ROOT = Path.cwd() / "DemoClinic运行目录"
IMPLEMENTATION = WORK_ROOT / "implementation"
IMPLEMENTATION.mkdir(parents=True, exist_ok=True)
_bundle = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_bundle))) as z:
    for info in z.infolist():
        destination = (IMPLEMENTATION / info.filename).resolve()
        if IMPLEMENTATION.resolve() not in destination.parents:
            raise ValueError("附带文件路径异常")
        destination.parent.mkdir(parents=True, exist_ok=True)
        if info.filename in ("体验Agent检查维度.md", "修复Agent执行规范.md"):
            # Preserve the user's editable protocol; bundled copy is only a fallback.
            editable = Path.cwd() / info.filename
            if not editable.exists():
                editable.write_bytes(z.read(info))
            destination.write_bytes(editable.read_bytes())
        else:
            destination.write_bytes(z.read(info))
if str(IMPLEMENTATION) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION))
import workflow_engine as wf
importlib.reload(wf)
print("运行目录：", WORK_ROOT)
print("实现代码：", IMPLEMENTATION)


## 2 · 安装并检查浏览器

首次运行需要联网下载 Python 依赖和 Chromium。之后会复用本机浏览器缓存。
浏览器在独立进程里运行，以兼容 Windows 下 Jupyter 的事件循环。

In [ ]:
import subprocess, importlib.util
for module, package in [("playwright", "playwright>=1.55,<2"), ("ipywidgets", "ipywidgets>=8,<9")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
print("浏览器和 ZIP 上传表单已就绪。若下一步表单无法显示，请重启 Jupyter 后重跑。")


## 3 · 配置模型和运行方式

这里设置工作流怎样运行，下一步填写待测项目的信息。运行本格只准备配置，不调用模型。

| 配置 | 含义 |
| --- | --- |
| `RUN_MODE` | `live` 为实际测试；`preview` 只检查网页预览。 |
| `analysis_model` / `test_model` / `fix_model` | 分析、体验、修复的模型型号。默认分别为 GLM-5.3、GLM-5.3-Flash、GLM-5.3。 |
| 各角色的 `provider` / `base_url` | 模型服务商和接口地址，需要与 API Key 配套。 |
| `vision` | 是否给体验模型发送截图，开启时模型须支持图片。 |
| `HEADLESS` | `False` 显示浏览器窗口，`True` 隐藏窗口。 |
| `experience_workers` | 默认同时执行2张独立任务；固定端口或网址模式自动串行。 |
| `history_batches` | 体验上下文保留的近期工具交互数量，默认2组。 |
| `prompt_path` / `repair_prompt_path` | 体验规范和修复规范的文件位置。 |

体验规范包含42个候选检查维度。分析 Agent 按项目生成任务卡，体验 Agent 只完成当前卡片，按需读取对应标准、保存检查证据和问题，不重复规划整个项目，也不负责写 HTML。

修复 Agent 只处理用户选择的问题，在源码副本中生成候选；是否采用修改，由第12步复测决定。

当前分层流程按阶段共用30分钟时限，不主动设置输出 token 或调用轮数上限。下方 `MAX_TEST_TURNS`、`MAX_FIX_TURNS`、`task_max_turns`、输出 token 预算等旧字段仅为兼容保留，不控制当前流程；服务商仍可能有自身输出限制。

修改模型配置后，要重新运行本格并在第4步重新提交项目，避免本轮继续使用旧配置。Key 在第5步输入，不要写进代码。仅需访问额外接口或模拟故障时，再配置对应域名和授权选项。


In [ ]:
# 工作流运行设置：平时可以保持默认，不在这里填写项目资料。
RUN_MODE = "live"          # live：调用模型；preview：只预览网页
HEADLESS = False          # False：显示浏览器窗口
MAX_TEST_TURNS = 60
MAX_FIX_TURNS = 14        # 每张所选问题卡片的模型调用上限
MAX_REPAIR_ISSUES = 1     # 旧版自动选卡上限；新版按用户选择逐条处理
EXTRA_ALLOWED_ORIGINS = []  # 项目必需的外部接口域名，按需配置

RUNTIME_CONFIG = {
    "layered_tasks": True,
    "experience_workers": 2,  # 最多同时工作的体验agent；网址/固定端口自动串行
    "max_analysis_tasks": 12,  # 分析agent最多生成12张任务卡
    "task_max_turns": 15,  # 每张卡片最多分配15轮，所有任务合计仍受MAX_TEST_TURNS限制
    'analysis_provider': "glm",
    'analysis_base_url': "https://open.bigmodel.cn/api/paas/v4",
    'analysis_model': "glm-5.3",  # 分析模型：独立配置与统计
    'analysis_max_output_tokens': 12000,  # 细化任务卡的输出预算
    'analysis_source_max_chars': 120000,  # 业务源码全文字符预算，超出停止
    'test_provider': "glm",
    'test_base_url': "https://open.bigmodel.cn/api/paas/v4",
    'test_model': "glm-5.3-flash",  # 体验模型：在这里修改型号
    'fix_provider': "glm",
    'fix_base_url': "https://open.bigmodel.cn/api/paas/v4",
    'fix_model': "glm-5.3",  # 修复模型：在这里修改型号
    'vision': True,
    'extra_allowed_origins': EXTRA_ALLOWED_ORIGINS,
    'headless': HEADLESS,
    'max_test_turns': MAX_TEST_TURNS,
    'max_fix_turns': MAX_FIX_TURNS,
    'max_repairs': MAX_REPAIR_ISSUES,
    'repair_prompt_path': str(Path.cwd() / "修复Agent执行规范.md"),
    'api_timeout': 120,
    'prompt_path': str(Path.cwd() / "体验Agent检查维度.md"),
    'browser_engine': "chromium",
    'max_tool_calls': 180,
    'history_batches': 2,          # 只携带最近2组工具交互摘要，较早证据通过read_memory读取
    'generate_report_content': False, # 默认由 Python 按证据生成简洁报告，不额外调用模型
    'max_report_turns': 10,         # 报告整理的独立调用预算
    'report_max_output_tokens': 6500,
    'max_test_seconds': 1800,
    'allow_fault_injection': False,
    'allow_permissions': False,
}
if RUN_MODE not in ("live", "preview"):
    raise ValueError("RUN_MODE 请填写 live 或 preview")
print("运行设置就绪。下一块填写本次提交的 demo 信息。")

for role, label in (("analysis", "分析"), ("test", "体验"), ("fix", "修复")):
    print(label, RUNTIME_CONFIG[role + "_model"], RUNTIME_CONFIG[role + "_base_url"])


## 4 · 上传源码 ZIP，填写项目信息

**本步不需要用户填写本机源码路径。**点击运行后会显示项目提交表单，点击“选择项目 ZIP”打开文件选择窗口，选中整个 demo 的源码压缩包。

### 提交流程

1. 将网站项目文件夹压缩为 ZIP，可带一层外部文件夹（如 GitHub 下载的项目包）。不要只打包一个入口而遗漏图片、样式或脚本。
2. 在下方表单选择 ZIP，填写项目信息。
3. 选择完整流程或仅出报告，确认项目允许在本机运行，点击“提交 ZIP 并准备项目”。
4. 看到“提交成功”后继续第 5 步。**表单出现不等于已提交，请逐格运行，不要直接运行全部。**

### 用户填写的维度

| 维度 | 填写内容与示例 |
|---|---|
| 项目名称、用途 | 例如“小组任务看板，帮助成员分配和跟踪课程任务” |
| 目标用户 | 例如“首次使用的大学生小组” |
| 核心任务与成功标准 | 至少一组、最多五组。例如“创建任务并刷新”；成功标准为“标题和日期均保留” |
| 已知问题与未完成功能 | 例如“通知暂未接入”，避免把已知情况当新发现 |
| 设计与操作限制 | 例如“保留现有布局，仅操作测试数据” |
| 重点测试范围 | 例如“桌面创建流程，手机宽度下的表单”；不意味着全部内容必然测完 |
| 账号与数据说明 | 说明测试账号角色、数据准备情况；不要填写密码；说明不会自动完成登录 |
| 额外验收要求 | 例如“截止日期不得早于开始日期”；不是自动执行的测试命令 |
| 服务方式 | 完整体验、修复与复测；或仅体验并出报告 |

### 网站怎样自动启动？

- **纯 HTML/CSS/JS**：识别 `index.html` 或唯一的 HTML 入口，用 Python 自带的网页服务启动，无需项目依赖。
- **简单 Python/Node 服务**：识别常见唯一入口及读取 `PORT` 的启动方式。Node 项目需本机已有 Node。
- **无法识别、固定端口或有依赖**：表单给出原因；展开“高级启动设置”，填写命令列表，例如 `["python", "app.py"]`。如果代码写死了 3000 端口，填固定端口 `3000`，并先关闭原来的服务。命令支持 `{python}`、`{port}` 占位符。
- 不会直接双击 ZIP 内的 `.cmd` 或猜着运行脚本。复杂数据库和远端服务还需要测试环境配置，本版并非任意项目的一键部署器。

### 文件会放在哪里？

每次提交建立新的 `DemoClinic运行目录/outputs/任务编号/`：`uploaded.zip` 保留上传包；`original` 存放解压后的原始项目；体验和修复使用另外的副本。重新提交不会覆盖上次任务。

ZIP 上限 50 MB，解压后上限 200 MB。依赖目录、凭证文件等会跳过并记录；不要上传真实私人数据。该版本在运行 Notebook 的电脑上执行项目，不是公开网站的服务器隔离环境。

用户资料会进入报告。API Key 在第 5 步输入。首次观察仍先看页面，再读取产品说明。


In [ ]:
# 运行本格显示表单。填写后点击“提交 ZIP 并准备项目”。
# 看到“提交成功”以后，再运行第 5 步；不要使用“运行全部”跳过表单提交。
import project_submission as intake
importlib.reload(intake)
SUBMISSION_FORM = intake.show_submission(RUNTIME_CONFIG, WORK_ROOT / "outputs", globals())


## 5 · 分别输入分析、体验、修复的 API Key

第三步分别配置三路模型的型号、接口地址和提供商。默认分析用 glm-5.3，体验用 glm-5.3-flash，修复用 glm-5.3；型号相同也仍按角色独立调用和统计。

依次输入分析、体验和修复 Key。仅体验并出报告时只输入前两路。可以复用有对应权限的同一个 Key；如果想在供应商后台也分开看用量，建议创建三个独立 Key（是否支持按 Key 筛选以平台为准）。

工作流会按角色汇总 API 返回的输入、输出 token；连接检查和本轮目录里的重复调用也会计入。密钥隐藏输入，不写入文件。第三方平台需先修改第三步接口地址。


In [ ]:
if "CONFIG" not in globals() or not CONFIG.get("submission_run"):
    raise RuntimeError("请先完成第 4 步上传，并点击提交直到显示提交成功。")
# 检查本次提交是否仍保留修改前的模型配置。
_model_fields = [role + "_" + field for role in ("analysis", "test", "fix")
                 for field in ("provider", "base_url", "model")] + ["vision"]
if any(CONFIG.get(field) != RUNTIME_CONFIG.get(field) for field in _model_fields):
    raise RuntimeError("模型配置已改变：请重新完成第四步提交，再继续，避免调用旧模型。")
from getpass import getpass
os.environ.pop("DEMO_ANALYSIS_KEY", None)
os.environ.pop("DEMO_TEST_KEY", None)
os.environ.pop("DEMO_FIX_KEY", None)
if RUN_MODE == "live":
    try:
        os.environ["DEMO_ANALYSIS_KEY"] = getpass("分析 agent 的 API Key：").strip()
        os.environ["DEMO_TEST_KEY"] = getpass("体验 agent 的 API Key：").strip()
        if CONFIG.get("repair_enabled"):
            os.environ["DEMO_FIX_KEY"] = getpass("修复 agent 的 API Key：").strip()
        if not os.environ.get("DEMO_ANALYSIS_KEY") or not os.environ.get("DEMO_TEST_KEY") or (CONFIG.get("repair_enabled") and not os.environ.get("DEMO_FIX_KEY")):
            raise ValueError("密钥不能为空")
    except BaseException:
        os.environ.pop("DEMO_ANALYSIS_KEY", None)
        os.environ.pop("DEMO_TEST_KEY", None)
        os.environ.pop("DEMO_FIX_KEY", None)
        raise
    print("分析、体验密钥已就绪；" + ("修复密钥已就绪。" if CONFIG.get("repair_enabled") else "本轮仅出报告，无需修复端 Key。"))
else:
    print("预览模式：不需要密钥，也不调用模型。")


## 6 · 检查接口可用性

各做一次最小文字请求，先发现 Key、余额、模型权限或网络问题，避免运行到中途才失败。
这一步验证文字连接；截图输入与工具调用会在真实体验阶段使用。

In [ ]:
if "CONFIG" not in globals() or not CONFIG.get("submission_run"):
    raise RuntimeError("请先完成第 4 步上传，并点击提交直到显示提交成功。")
# 检查本次提交是否仍保留修改前的模型配置。
_model_fields = [role + "_" + field for role in ("analysis", "test", "fix")
                 for field in ("provider", "base_url", "model")] + ["vision"]
if any(CONFIG.get(field) != RUNTIME_CONFIG.get(field) for field in _model_fields):
    raise RuntimeError("模型配置已改变：请重新完成第四步提交，再继续，避免调用旧模型。")
if RUN_MODE == "live":
    for role in (("analysis", "test", "fix") if CONFIG.get("repair_enabled") else ("analysis", "test")):
        client = wf.ModelClient(dict(CONFIG, usage_log_path=str(RUN / "connection_usage.jsonl")), role)
        print("准备检查：", {"analysis":"分析", "test":"体验", "fix":"修复"}[role], client.model, client.base_url)
        answer = client.request([{"role":"user", "content":"请只回复：连接成功。"}])
        print({"analysis":"分析接口", "test":"体验接口", "fix":"修复接口"}[role], client.model, client.base_url, "已响应；用量：", client.usage)
else:
    print("已跳过真实模型连接检查。")
API_USAGE = wf.show_usage(RUN, CONFIG)


## 7 · 自动启动项目并预览

启动用户提交的源码副本，确认网页能打开，并显示截图。此步不调用模型。启动失败时查看本轮目录的 server.log，调整第四步启动设置后重新提交。


In [ ]:
if "CONFIG" not in globals() or not CONFIG.get("submission_run"):
    raise RuntimeError("请先完成第 4 步上传，并点击提交直到显示提交成功。")
RUN = Path(CONFIG["submission_run"])
# 每次重新预览使用新目录，不覆盖旧截图。
PREVIEW_FOLDER = RUN / ("preview_" + datetime.datetime.now().strftime("%H%M%S_%f"))
CONFIG = dict(CONFIG)
wf.dump(RUN / "config.json", CONFIG)

PREVIEW = wf.target_config(CONFIG, CONFIG.get("source_path"), PREVIEW_FOLDER)
with wf.RunningTarget(PREVIEW) as url:
    preview = wf.worker({"operation":"replay", "url":url,
        "actions":[{"action":"goto","url":url}], "checks":[], "output":PREVIEW["folder"]},
        PREVIEW["folder"], CONFIG, credentials=False)
from IPython.display import Image
display(Image(filename=preview["screenshot"]))
print("本轮输出：", RUN)

## 8 · 生成12张分层任务卡

分析 Agent 根据首页、项目根目录 README 和用户填写的资料生成任务，不读取业务源码。程序校验任务数量、必填内容和依赖关系，通过后保存 `task_plan.json`。

每张任务卡说明检查范围、准备方法、测试输入、操作步骤、预期结果和停止条件。看完计划后，再运行第9步。

模型请求按阶段共用30分钟时限。推理文本、正文和工具调用显示在独立日志窗口，并保存到本轮 `agent_logs`。超时或输出不完整时保留已收到的记录，不执行残缺计划。关闭日志窗口不会停止任务。


In [ ]:
CONFIG["analysis_timeout_seconds"] = 20 * 60
CONFIG["agent_log_popup"] = True
CONFIG["analysis_length_retries"] = 0
importlib.reload(wf)
import task_queue as tq
importlib.reload(tq)
if RUN_MODE == "live":
    PLAN = tq.plan_only(CONFIG, RUN)
    print("任务卡已保存：", RUN / "task_plan.json")
    print(json.dumps(PLAN, ensure_ascii=False, indent=2))
    API_USAGE = wf.show_usage(RUN, CONFIG)
else:
    print("预览模式：未调用分析模型，未生成任务卡。")


## 9 · 执行已生成的体验任务

确认上一步任务计划后，单独运行本格。它读取本轮 `task_plan.json`，不再调用分析 agent。

Python 校验计划并按依赖分发任务；前置任务失败、受阻或未完成时，下游阻塞，独立任务继续。每张卡使用独立源码副本和新浏览器，普通源码项目最多并发 2 个，网址模式和固定端口串行。

体验预算仍使用第三步配置。每次执行创建新的 `experience_execution_*` 目录保留证据，并更新 `task_state.json` 和 `baseline.json`。重复运行本格会重新体验，但不会重新分析。

体验 agent 只执行当前卡片：使用本卡维度标准，直接进入核心任务，保存检查证据和问题小卡片；不重复全站分析、不走六阶段巡检、不生成全局报告。42维标准保持原样。


In [ ]:
importlib.reload(wf)
import task_queue as tq
importlib.reload(tq)
BASELINE = tq.execute_plan(CONFIG, RUN) if RUN_MODE == "live" else []
if RUN_MODE == "preview": print("预览模式：没有生成虚构的体验结果。")
for result in BASELINE:
    print(result["persona"]["name"], ":", result["stop_reason"], "; 检查数:", len(result["checks"]))
API_USAGE = wf.show_usage(RUN, CONFIG)


## 10 · 独立重放问题，生成第一版图文报告

问题发现后，程序在全新副本上重放操作。只有原步骤可执行且正确预期仍不满足，才标为“已复现（按所填预期）”。
主观建议、工具定位失败和无法重现的问题不会直接触发自动修复。预期本身是否合理仍取决于你填写的产品约定。
默认由 Python 根据结构化问题与原始证据直接排版，不额外调用模型写 HTML，生成 `体验与修复报告.html` 和 `修复问题单.json`。复现状态以程序结果为准，疑似问题和体验建议不会自动进入修复队列。

只需要报告时，到本步即可结束：下方提供 HTML 报告和 JSON 问题单链接，无需运行第11、12步；使用完可运行第14步清除密钥。


In [ ]:
importlib.reload(wf)
ISSUES = wf.reproduce_issues(CONFIG, RUN, BASELINE) if RUN_MODE == "live" else []
REPORT = wf.render_report(RUN, CONFIG, BASELINE, ISSUES)
for issue in ISSUES: print(issue["id"], issue["title"], "→", issue["status"])
display(Markdown(f"第一版图文报告已生成：`{REPORT}`"))
from IPython.display import FileLink
for report_file in (REPORT, RUN / "修复问题单.json"):
    display(FileLink(str(report_file.resolve().relative_to(Path.cwd().resolve())).replace("\\", "/")))


## 11 · 每轮选择一个问题，只生成修复候选

查看第10步报告后，输入一个问题编号。本步只调用修复 Agent 生成候选，不执行浏览器复测，也不宣称修复成功。

候选生成后必须运行第12步。第12步通过才累计采用本轮修改；未复测前不允许开始下一轮。之后可再次运行第11步选择下一问题，下一轮基于已验证副本继续修复。失败候选不覆盖此前成果，原始项目保持不变。

留空或重复选择已修复编号不会调用修复模型。修复阶段共用30分钟时限，不设输出 token 或调用轮数上限。


In [ ]:
importlib.reload(wf)
ISSUES = wf.load(RUN / "issues.json")
_saved_repair = wf.load(RUN / "repair.json") if (RUN / "repair.json").exists() else {}
_fixed_ids = set(_saved_repair.get("verification", {}).get("verified_ids", []))
ELIGIBLE_ISSUES = [item for item in ISSUES if wf.repairable(CONFIG, item)]
if RUN_MODE == "live" and CONFIG.get("repair_enabled", False) and ELIGIBLE_ISSUES:
    for item in ISSUES:
        state = "已修复并验证通过" if item["id"] in _fixed_ids else ("可修复" if wf.repairable(CONFIG, item) else "暂不修复：" + item.get("status", "未确认"))
        print(item["id"], item["title"], "—", state)
    selected_text = input("只输入一个要修复的问题编号（例如1），留空跳过：")
    SELECTED_ISSUE_IDS = wf.select_repair_issues(selected_text, CONFIG, ISSUES)
    REPAIR = wf.repair_selected(CONFIG, RUN, BASELINE, ISSUES, SELECTED_ISSUE_IDS)
else:
    reason = "没有已复现且可修复的问题，跳过修复" if not ELIGIBLE_ISSUES else "修复未开启或为预览模式"
    REPAIR = {"status":"skipped", "reason":reason, "verification":{"status":"skipped", "reason":reason}}
print(REPAIR.get("summary", REPAIR.get("reason", "")))
for item in REPAIR.get("issue_results", []):
    print(item["id"], item["status"], item["summary"])
API_USAGE = wf.show_usage(RUN, CONFIG)


## 12 · 复测问题并更新报告

这是唯一的修复后复测步骤。不调用模型，只读取本轮保存的候选、问题卡片和成功检查记录。

依次检查本轮所选问题、此前已修复问题、已有正常流程及 Python/JavaScript 文件语法。所有必要检查通过才采用候选；失败时保留此前已验证版本，没有此前版本则保留原始项目。缺少 Node.js 或需要尚未支持的编译器时，不把语法检查算作通过。

- `verified`：所选修改通过本次检查，不代表整个项目没有其他缺陷。
- `partial_verified`：本轮未全部通过，保留此前已验证的修改。
- `verification_blocked`：操作或检查无法执行，暂时不能判断修复是否有效。
- `not_verified`：已执行的检查未满足预期，或没有足够的成功回归检查点。

ID 改变时使用已记录的控件身份和备用定位；同名按钮、列表重排等有歧义时停止，不猜测目标。断言保留原比较方式，异步结果默认最多等待2秒，可通过 `CONFIG["assertion_timeout_ms"]` 调整（最多10秒）。复测执行默认总时限600秒，可通过 `CONFIG["verification_timeout_seconds"]` 调整。

程序核对原始源码、候选、问题及检查记录的内容指纹。旧记录首次复测时建立指纹，只能保护此后的变更，不能证明更早的文件未被改动。报告或导出异常时恢复已保存的发布信息；交付目录先完整生成，再替换。强制结束内核后若残留 `.verification.lock`，确认没有复测进程在运行后再删除该锁文件。

没有候选时跳过复测。已有本轮记录时可以单独运行本步；更新 Notebook 代码包后需先运行第1步解包，不必重新调用模型体验。


In [ ]:
importlib.reload(wf)  # 载入最新复测逻辑，沿用当前任务和候选代码
REPAIR, VERIFICATION, REPORT = wf.reverify_saved(CONFIG, RUN)
display(HTML("<h3>最终状态：" + VERIFICATION["status"] + "</h3>"))
print(VERIFICATION.get("reason", ""))
import html
display(HTML('<iframe style="width:100%;height:800px;border:1px solid #ddd;border-radius:12px" srcdoc="'
             + html.escape(REPORT.read_text(encoding="utf-8"), quote=True) + '"></iframe>'))
API_USAGE = wf.show_usage(RUN, CONFIG)


## 13 · 获取报告与完整项目 ZIP

运行本格，将结果集中整理到本次任务的 **`交付结果`** 文件夹，并在 Notebook 显示可打开或下载的链接。

- **体验与修复报告.html**：给用户阅读，图片已内嵌，可以离线打开。
- **修复问题单.json**：结构化的问题与证据记录。
- **项目 ZIP**：保留源项目的图片、字体、样式和其他资源，覆盖采用的代码修改。通过复测叫“修复后的项目”；有修改但未通过叫“待验证的修改候选”；没有修改则叫“原始项目_未修改”。
- **修改与运行说明.txt**：记录修改文件、启动方式和复测结果。
- 其他复测记录与代码差异按本轮实际结果提供。

原始上传包不会被覆盖。项目 ZIP 不包含被跳过的依赖和凭证目录；使用者仍需按说明准备必要环境。报告中的通过仅针对本次选择的修复与检查，不是整个网站的质量保证。


In [ ]:
# 生成面向用户的报告与项目压缩包。
RESULTS = intake.export_project(RUN, CONFIG, REPAIR, VERIFICATION)
print("本轮任务目录：", RUN)
print("报告和项目 ZIP：", RESULTS)
from IPython.display import FileLink
for result_file in sorted(RESULTS.iterdir()):
    try:
        relative = result_file.resolve().relative_to(Path.cwd().resolve())
        display(FileLink(str(relative).replace("\\", "/")))
    except ValueError:
        print(result_file)
if VERIFICATION.get("status") != "verified":
    print("本轮未达到修复验证通过状态。请查看报告，勿把候选修改当成已修复。")


## 14 · 清除本次密钥

运行本格，清除进程中保存的三路 API Key。已有报告和项目文件不受影响。之后若要再次调用模型，回到第5步重新输入。


In [ ]:
for name in ("DEMO_ANALYSIS_KEY", "DEMO_TEST_KEY", "DEMO_FIX_KEY"):
    os.environ.pop(name, None)
print("内存中的三路密钥已清除。")


## 文件与结果

每次上传都会创建独立运行目录，包含原始 ZIP、项目副本、检查证据和报告。第10步提供 HTML 与 JSON，第13步整理报告和项目 ZIP。

原始源码不会被修复覆盖。只有通过第12步检查的修改才累计采用；未测试、未复现和执行受阻会分别记录。
